In [ ]:
import tensorflow as tf
import numpy as np
import os

# Enable memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

print("GPU Ready:", tf.config.list_physical_devices('GPU'))


GPU Ready: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import tensorflow as tf
import numpy as np
import os
import glob

# GPU memory safety
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

print("GPU:", tf.config.list_physical_devices('GPU'))


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

data_dir = os.path.join(path, "chest_xray")
train_dir = os.path.join(data_dir, "train")


Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.


In [ ]:
normal_paths = glob.glob(train_dir + "/NORMAL/*.jpeg")
pneumonia_paths = glob.glob(train_dir + "/PNEUMONIA/*.jpeg")

all_paths = normal_paths + pneumonia_paths
labels = [0]*len(normal_paths) + [1]*len(pneumonia_paths)

all_paths = np.array(all_paths)
labels = np.array(labels)

print("Total images:", len(all_paths))


Total images: 5216


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label


In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_model(base_model):

    base_model.trainable = False  # Stage 1

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x, training=True)  # Monte Carlo Dropout
    output = Dense(1, activation='sigmoid')(x)

    model = Model(base_model.input, output)

    model.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
import tensorflow as tf
import numpy as np
import os
import glob

# GPU safety
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

print("GPU:", tf.config.list_physical_devices('GPU'))


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

data_dir = os.path.join(path, "chest_xray")
train_dir = os.path.join(data_dir, "train")


Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.


In [ ]:
normal_paths = glob.glob(train_dir + "/NORMAL/*.jpeg")
pneumonia_paths = glob.glob(train_dir + "/PNEUMONIA/*.jpeg")

all_paths = normal_paths + pneumonia_paths
labels = [0]*len(normal_paths) + [1]*len(pneumonia_paths)

all_paths = np.array(all_paths)
labels = np.array(labels)

print("Total:", len(all_paths))
print("Normal:", len(normal_paths))
print("Pneumonia:", len(pneumonia_paths))


Total: 5216
Normal: 1341
Pneumonia: 3875


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label


In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

def build_model(base_model):

    base_model.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.4)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(base_model.input, output)

    model.compile(
        optimizer=Adam(5e-5),
        loss=BinaryCrossentropy(label_smoothing=0.05),
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model


In [ ]:
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

data_dir = os.path.join(path, "chest_xray")
train_dir = os.path.join(data_dir, "train")


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import glob
import numpy as np
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

data_dir = os.path.join(path, "chest_xray")
train_dir = os.path.join(data_dir, "train")

normal_paths = glob.glob(train_dir + "/NORMAL/*.jpeg")
pneumonia_paths = glob.glob(train_dir + "/PNEUMONIA/*.jpeg")

all_paths = normal_paths + pneumonia_paths
labels = [0]*len(normal_paths) + [1]*len(pneumonia_paths)

all_paths = np.array(all_paths)
labels = np.array(labels)

print("Normal:", len(normal_paths))
print("Pneumonia:", len(pneumonia_paths))

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Normal: 1341
Pneumonia: 3875


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label


In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import AdamW

def build_model(base_model, steps_per_epoch):

    base_model.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.4)(x)
    output = Dense(1, activation='sigmoid')(x)

    model = Model(base_model.input, output)

    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=3e-4,
        decay_steps=steps_per_epoch * 5
    )

    optimizer = AdamW(
        learning_rate=lr_schedule,
        weight_decay=1e-4
    )

    model.compile(
        optimizer=optimizer,
        loss=BinaryCrossentropy(label_smoothing=0.05),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.utils import class_weight
from tensorflow.keras.applications import EfficientNetB0, DenseNet121
from tensorflow.keras.callbacks import EarlyStopping

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_aucs = []

# Initialize global variables to store results from the last fold
final_val_labels = None
final_ensemble_pred = None

for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, labels)):

    print(f"\n========== Fold {fold+1} ==========")

    train_paths, val_paths = all_paths[train_idx], all_paths[val_idx]
    train_labels, val_labels = labels[train_idx], labels[val_idx]

    class_weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels),
        y=train_labels
    )
    class_weights = dict(enumerate(class_weights))

    train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))

    train_ds = train_ds.map(load_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    steps_per_epoch = len(train_paths) // BATCH_SIZE

    # -------- EfficientNet --------
    base_eff = EfficientNetB0(weights='imagenet', include_top=False,
                              input_shape=(224,224,3))
    model_eff = build_model(base_eff, steps_per_epoch)

    model_eff.fit(train_ds,
                  validation_data=val_ds,
                  epochs=5,
                  class_weight=class_weights,
                  callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
                  verbose=1)

    pred_eff = model_eff.predict(val_ds)
    tf.keras.backend.clear_session()

    # -------- DenseNet --------
    base_den = DenseNet121(weights='imagenet', include_top=False,
                           input_shape=(224,224,3))
    model_den = build_model(base_den, steps_per_epoch)

    model_den.fit(train_ds,
                  validation_data=val_ds,
                  epochs=5,
                  class_weight=class_weights,
                  callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
                  verbose=1)

    pred_den = model_den.predict(val_ds)

    # -------- Weighted Ensemble --------
    auc_eff = roc_auc_score(val_labels, pred_eff)
    auc_den = roc_auc_score(val_labels, pred_den)

    w_eff = auc_eff / (auc_eff + auc_den)
    w_den = auc_den / (auc_eff + auc_den)

    ensemble_pred = w_eff * pred_eff + w_den * pred_den
    ensemble_auc = roc_auc_score(val_labels, ensemble_pred)

    fold_aucs.append(ensemble_auc)

    print("EfficientNet AUC:", auc_eff)
    print("DenseNet AUC:", auc_den)
    print("Ensemble AUC:", ensemble_auc)

    tf.keras.backend.clear_session()

    # Store results from the last fold globally
    if fold == skf.n_splits - 1:
        final_val_labels = val_labels
        final_ensemble_pred = ensemble_pred

print("\nMean AUC:", np.mean(fold_aucs))


========== Fold 1 ==========
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 114s 253ms/step - accuracy: 0.5259 - auc: 0.4874 - loss: 0.8638 - val_accuracy: 0.7423 - val_auc: 0.5000 - val_loss: 0.6324
Epoch 2/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 36s 89ms/step - accuracy: 0.4380 - auc: 0.4397 - loss: 1.1902 - val_accuracy: 0.7423 - val_auc: 0.5000 - val_loss: 0.6060
Epoch 3/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 36s 91ms/step - accuracy: 0.3802 - auc: 0.3651 - loss: 1.2023 - val_accuracy: 0.7423 - val_auc: 0.5000 - val_loss: 0.5837
Epoch 4/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 40s 88ms/step - accuracy: 0.2986 - auc: 0.3373 - loss: 1.1686 - val_accuracy: 0.7423 - val_auc: 0.5000 - val_loss: 0.6281
Epoch 5/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 40s 86ms/step - accuracy: 0.3816 - auc: 0.3689 - loss: 1.0278 - val_accuracy: 0.7423 - val_auc: 0.5000 - val_loss: 0.6916
66/66 ━━━━━━━━━━━━━━━━━━━━ 20s 191ms/step
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
261/261 ━